In [3]:
# --- 1. Imports ---
import os
import numpy as np
import cv2
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

print("Libraries imported.")

AttributeError: module 'numpy.linalg._umath_linalg' has no attribute '_ilp64'

In [ ]:
# --- 2. Load Movie Template ---
output_dir = "../Dataset/RAW"
manifest_path = os.path.join(output_dir, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)

print("Loading Natural Movie One Template...")
movie_template = cache.get_natural_movie_template(1)
print(f"Movie Template Shape: {movie_template.shape}")
print(f"Data Type: {movie_template.dtype}")
print(f"Min: {movie_template.min()}, Max: {movie_template.max()}")

In [ ]:
# --- 3. Generate Video ---
save_dir = "../Results/Movies"
os.makedirs(save_dir, exist_ok=True)
video_path = os.path.join(save_dir, "Natural_Movie_One.mp4")

# Prepare Frames
frames = movie_template

# Normalize to 0-255 uint8 if necessary
if frames.dtype != np.uint8:
    print("Normalizing frames to uint8...")
    frames_norm = cv2.normalize(frames, None, 0, 255, cv2.NORM_MINMAX)
    frames_uint8 = frames_norm.astype(np.uint8)
else:
    frames_uint8 = frames

height, width = frames_uint8.shape[1], frames_uint8.shape[2]
fps = 30 # Standard frame rate for this stimulus

# Initialize Video Writer
# 'mp4v' is a widely supported codec for .mp4
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
out = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

print(f"Writing {len(frames_uint8)} frames to {video_path}...")

for i in range(len(frames_uint8)):
    frame = frames_uint8[i]
    # Convert grayscale to BGR (OpenCV expects BGR for color video writers)
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    out.write(frame_bgr)

out.release()
print("Video generation complete.")